In [1]:
import pandas as pd
import torch

# Per riproducibilità
torch.manual_seed(1234)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

df = pd.read_csv("../data/all_ships.csv")


In [2]:
from sklearn.model_selection import train_test_split

X = df[["sex", "age", "age_missing", "class", "crew"]]
Y = df["survived"]

X_tensor = torch.tensor(X.values, dtype=torch.float32)
Y_tensor = torch.tensor(Y.values, dtype=torch.long) # La CrossEntropyLoss richiede target long

# per riproducibilità si usa random_state fissato
X_train, X_test, Y_train, Y_test = train_test_split(X_tensor, Y_tensor, test_size=0.2, random_state=42, stratify=Y_tensor)

mean = X_train.mean(0)
std  = X_train.std(0)

X_train_norm = (X_train - mean) / std
X_test_norm  = (X_test  - mean) / std


In [3]:
from torch.utils.data import TensorDataset, DataLoader

train_ds = TensorDataset(X_train_norm, Y_train)
test_ds = TensorDataset(X_test_norm, Y_test)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64)


In [4]:
from torch import nn

class MLP_2Layer(nn.Module):
    """
    Semplice MLP con:
        - input_dim = 5 (feature)
        - hidden_dim = 3 (layer nascosto)
        - output_dim = 2 (classi: morto / sopravvissuto)
    Architettura:
        input -> Linear(5,3) -> ReLU -> Linear(3,2) -> logits
    """
    def __init__(self, in_features, hidden_dim, out_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_dim), # rappresentazione nascosta (batch_size, 3)
            nn.ReLU(), # funzione di attivazione
            nn.Linear(hidden_dim, out_features) # punteggi non normalizzati (batch_size, 2)
        )

    def forward(self, x):
        return self.net(x) 


In [5]:
class DeepMLP(nn.Module):
    """
    Modello MLP profondo definito dinamicamente.
    hidden_units: lista es. [8, 4] crea due hidden layer 5->8->4->2
    """
    def __init__(self, in_features, hidden_units, out_features):
        super().__init__()
        
        # costruiamo i layer uno dopo l'altro, aggiungendo ReLU tra di essi
        layers = []
        input_dim = in_features
        for hidden_dim in hidden_units:
            layers.append(nn.Linear(input_dim, hidden_dim))
            layers.append(nn.ReLU())
            input_dim = hidden_dim

        # layer di output (logits)
        layers.append(nn.Linear(input_dim, out_features))

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


In [6]:
from torch.optim import SGD
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import accuracy_score

def train_model(model, train_loader, test_loader, lr=0.05, epochs=300):
    writer = SummaryWriter(f'../results/{model._get_name()}')
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=0.001)

    for epoch in range(epochs):
        model.train()

        train_loss = 0.0
        y_true = []
        y_pred = []
        
        for X_batch, Y_batch in train_loader:
            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)

            output = model(X_batch)
            loss = criterion(output, Y_batch)

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            train_loss += loss.item() * X_batch.size(0)

            # l'output è array di logits quindi si prende il punteggio più alto che indica la classe più probabile
            preds = output.argmax(dim=1) 
            y_true.extend(Y_batch.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

        train_loss /= len(train_loader.dataset)
        train_acc = accuracy_score(y_true, y_pred)

        model.eval()

        test_loss = 0.0
        y_true = []
        y_pred = []

        with torch.no_grad():
            for X_batch, Y_batch in test_loader:
                X_batch = X_batch.to(device)
                Y_batch = Y_batch.to(device)

                output = model(X_batch)

                loss = criterion(output, Y_batch)
                test_loss += loss.item() * X_batch.size(0)
                
                preds = output.argmax(dim=1)
                y_true.extend(Y_batch.cpu().numpy())
                y_pred.extend(preds.cpu().numpy())

        test_loss /= len(test_loader.dataset)
        test_acc = accuracy_score(y_true, y_pred)

        writer.add_scalar('loss/train', train_loss, epoch)
        writer.add_scalar('accuracy/train', train_acc, epoch)
        writer.add_scalar('loss/test', test_loss, epoch)
        writer.add_scalar('accuracy/test', test_acc, epoch)

        if epoch % 50 == 0:
            print(f"Epoch {epoch+1}/{epochs} | train_loss {train_loss:.4f} | train_acc {train_acc:.4f} | test_loss {test_loss:.4f} | test_acc {test_acc:.4f}")

    print(f"Epoch {epoch+1}/{epochs} | train_loss {train_loss:.4f} | train_acc {train_acc:.4f} | test_loss {test_loss:.4f} | test_acc {test_acc:.4f}")
    writer.close()

    return model, train_loss, train_acc, test_loss, test_acc

In [7]:
# Addestramento modello base
base_model = MLP_2Layer(in_features=5, hidden_dim=3, out_features=2)
base_model, mlp_train_loss, mlp_train_acc, mlp_test_loss, mlp_test_acc = train_model(base_model, train_loader, test_loader)


# Addestramento modello deep
deep_model = DeepMLP(in_features=5, hidden_units=[8, 4], out_features=2)
deep_model, deep_train_loss, deep_train_acc, deep_test_loss, deep_test_acc = train_model(deep_model, train_loader, test_loader)


Epoch 1/300 | train_loss 0.6264 | train_acc 0.6755 | test_loss 0.6105 | test_acc 0.6788
Epoch 51/300 | train_loss 0.5553 | train_acc 0.7139 | test_loss 0.5554 | test_acc 0.7074
Epoch 101/300 | train_loss 0.5553 | train_acc 0.7135 | test_loss 0.5466 | test_acc 0.7139
Epoch 151/300 | train_loss 0.5497 | train_acc 0.7168 | test_loss 0.5492 | test_acc 0.7100
Epoch 201/300 | train_loss 0.5504 | train_acc 0.7165 | test_loss 0.5523 | test_acc 0.7126
Epoch 251/300 | train_loss 0.5553 | train_acc 0.7165 | test_loss 0.5488 | test_acc 0.7152
Epoch 300/300 | train_loss 0.5490 | train_acc 0.7122 | test_loss 0.5505 | test_acc 0.7087
Epoch 1/300 | train_loss 0.6262 | train_acc 0.6553 | test_loss 0.5826 | test_acc 0.6749
Epoch 51/300 | train_loss 0.5487 | train_acc 0.7207 | test_loss 0.5486 | test_acc 0.7178
Epoch 101/300 | train_loss 0.5500 | train_acc 0.7178 | test_loss 0.5578 | test_acc 0.7152
Epoch 151/300 | train_loss 0.5498 | train_acc 0.7168 | test_loss 0.5408 | test_acc 0.7178
Epoch 201/300 | 

In [8]:
print("Model Comparison\n")

print("MLP_2Layer")
print(f"train_loss: {mlp_train_loss:.4f}")
print(f"train_acc: {mlp_train_acc:.4f}")
print(f"test_loss: {mlp_test_loss:.4f}")
print(f"test_acc: {mlp_test_acc:.4f}\n")

print("DeepMLP")
print(f"train_loss: {deep_train_loss:.4f}")
print(f"train_acc: {deep_train_acc:.4f}")
print(f"test_loss: {deep_test_loss:.4f}")
print(f"test_acc: {deep_test_acc:.4f}")

Model Comparison

MLP_2Layer
train_loss: 0.5490
train_acc: 0.7122
test_loss: 0.5505
test_acc: 0.7087

DeepMLP
train_loss: 0.5465
train_acc: 0.7126
test_loss: 0.5482
test_acc: 0.7152
